# Final GPU acceptance — English, Hindi, and experimental Odia V2
Run one cell at a time on a fresh Colab T4. Expensive route results are saved immediately. Nothing here trains a foundation model.

In [ ]:
import os, pathlib, psutil, subprocess, sys
gpu=subprocess.run(['nvidia-smi'],capture_output=True,text=True)
assert gpu.returncode==0, 'Select Runtime → Change runtime type → T4 GPU, then reconnect.'
print(gpu.stdout); print('system Python',sys.version); print('RAM GiB',round(psutil.virtual_memory().total/2**30,2))

## 2. Upload and extract `finalpass.tar.gz` only when the project is absent

In [ ]:
from pathlib import Path
PROJECT=Path('/content/voice-translator')
if not (PROJECT/'pyproject.toml').exists():
    from google.colab import files
    print('Upload finalpass.tar.gz from Windows Downloads')
    uploaded=files.upload(); archive=Path('/content')/next(iter(uploaded))
    PROJECT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['tar','-xzf',str(archive),'-C',str(PROJECT)],check=True)
print('Project:',PROJECT)

## 3–5. System packages, isolated environments, and pinned dependencies

In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','libsndfile1','python3.12-venv'],check=True)
MAIN=Path('/content/venvs/main'); OMNI=Path('/content/venvs/omnivoice')
subprocess.run([sys.executable,'-m','venv',str(MAIN)],check=True)
PY=str(MAIN/'bin/python'); PIP=[PY,'-m','pip']
subprocess.run(PIP+['install','-q','--upgrade','pip','wheel','setuptools'],check=True)
subprocess.run(PIP+['install','-q','-r',str(PROJECT/'requirements-gpu.txt'),'-r',str(PROJECT/'requirements-odia.txt')],check=True)
print('Every project command below uses',PY)

## 6. Validate dependency imports and checkpoint access
IndicConformer is gated. Accept AI4Bharat’s Hugging Face access terms and set `HF_TOKEN` in Colab Secrets if the check reports an access error.

In [ ]:
check=subprocess.run([PY,'-c','import torch, transformers, faster_whisper, qwen_tts; import nemo.collections.asr; from IndicTransToolkit.processor import IndicProcessor; print(torch.cuda.is_available())'],capture_output=True,text=True)
print(check.stdout,check.stderr); assert check.returncode==0
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
except Exception: token=os.environ.get('HF_TOKEN','')
download_env=os.environ.copy()
if token: download_env['HF_TOKEN']=token
models=['Systran/faster-whisper-small','facebook/seamless-m4t-v2-large','Qwen/Qwen3-TTS-12Hz-0.6B-Base','ai4bharat/indicconformer_stt_or_hybrid_ctc_rnnt_large','ai4bharat/indictrans2-indic-en-dist-200M']
download_code='from huggingface_hub import snapshot_download; import sys\nfor model in sys.argv[1:]:\n print("Downloading/validating",model,flush=True); path=snapshot_download(model); print("validated",path,flush=True)'
subprocess.run([PY,'-c',download_code,*models],env=download_env,check=True)
print('All required checkpoint snapshots validated in the shared cache.')

## 7. Upload separate English, Hindi, and Odia WAV recordings

In [ ]:
from google.colab import files
INPUTS=Path('/content/inputs'); INPUTS.mkdir(exist_ok=True)
paths={}
for language,name in [('en','english.wav'),('hi','hindi.wav'),('ory','odia.wav')]:
    print('Upload',name); uploaded=files.upload(); source=Path('/content')/next(iter(uploaded)); target=INPUTS/name; target.write_bytes(source.read_bytes()); paths[language]=target
print(paths)

## 8. Run lightweight tests before downloading model weights

In [ ]:
subprocess.run([PY,'-m','pip','install','-q','-r',str(PROJECT/'requirements-lightweight.txt')],check=True)
subprocess.run([PY,'-m','pytest','-ra',str(PROJECT/'tests')],cwd=PROJECT,check=True)

## 9–12. One persistent process: English routes, Hindi routes, Odia direct baseline, then Odia V2
The runner saves `summary.partial.json` after every route, uses sentence-level Qwen generation/retry, and loads reusable models only once.

In [ ]:
from datetime import datetime, timezone
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); EVIDENCE=Path('/content')/f'voice-translator-evidence-{STAMP}'; EVIDENCE.mkdir()
env=os.environ.copy(); env.update({'VT_WHISPER_MODEL':'small','VT_WHISPER_DEVICE':'cuda','VT_WHISPER_COMPUTE_TYPE':'float16','VT_SEAMLESS_MODEL':'facebook/seamless-m4t-v2-large','VT_SEAMLESS_DEVICE':'cuda','VT_SEAMLESS_DTYPE':'float16','VT_QWEN_DEVICE':'cuda:0','VT_QWEN_DTYPE':'float32','VT_QWEN_ATTENTION':'eager','VT_QWEN_SEED':'42'})
command=[PY,str(PROJECT/'scripts/run_final_acceptance.py'),'--english',str(paths['en']),'--hindi',str(paths['hi']),'--odia',str(paths['ory']),'--output',str(EVIDENCE)]
print('Starting real models; this can take a long time:',command)
run=subprocess.run(command,cwd=PROJECT,env=env,text=True)
print('runner exit',run.returncode,'partial evidence remains at',EVIDENCE)

## Display every completed output and intermediate text

In [ ]:
import json
from IPython.display import Audio,display
summary_path=EVIDENCE/'summary.json' if (EVIDENCE/'summary.json').exists() else EVIDENCE/'summary.partial.json'
summary=json.loads(summary_path.read_text()); display(summary)
for wav in sorted(EVIDENCE.glob('*.wav')):
    print(wav.name); display(Audio(str(wav)))

## 13. Optional isolated OmniVoice Odia A/B
Installation or inference failure is recorded as a skip and does not invalidate Qwen outputs. OmniVoice remains experimental and never replaces Qwen automatically.

In [ ]:
subprocess.run([sys.executable,'-m','venv',str(OMNI)],check=True)
OMNI_PY=str(OMNI/'bin/python'); install=subprocess.run([OMNI_PY,'-m','pip','install','-q','-r',str(PROJECT/'requirements-omnivoice.txt')],capture_output=True,text=True)
status={'status':'skipped','reason':install.stderr.strip() or 'installation failed'}
if install.returncode==0:
    odia_results=[r for r in summary.get('routes',[]) if r.get('source')=='ory' and r.get('status')=='completed']
    for route in odia_results:
        result=route['result']; target=result['target_language']; status_path=EVIDENCE/f'omnivoice-{target}.json'
        subprocess.run([PY,str(PROJECT/'scripts/run_omnivoice_ab.py'),'--python',OMNI_PY,'--reference-audio',str(paths['ory']),'--reference-text',result['source_transcript'],'--text',result['translated_text'],'--output',str(EVIDENCE/f'omnivoice-{target}.wav'),'--status',str(status_path)],cwd=PROJECT)
else: (EVIDENCE/'omnivoice-install-skip.json').write_text(json.dumps(status,indent=2))
print('OmniVoice step complete; inspect its JSON status files.')

## 14. Produce comparison report without invented listening scores

In [ ]:
report={'owner_listening_scores':'not measured','summary':summary,'omnivoice':[json.loads(p.read_text()) for p in EVIDENCE.glob('omnivoice-*.json')]}
(EVIDENCE/'comparison-report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(EVIDENCE/'comparison-report.json')

## 15. Build the frontend with the environment’s explicit executables

In [ ]:
subprocess.run(['npm','ci'],cwd=PROJECT/'frontend',check=True)
subprocess.run(['npm','test'],cwd=PROJECT/'frontend',check=True)
subprocess.run(['npm','run','typecheck'],cwd=PROJECT/'frontend',check=True)
subprocess.run(['npm','run','build'],cwd=PROJECT/'frontend',check=True)

## 16–17. Owner-executed only: serve FastAPI + built microphone UI and start randomized temporary tunnel
This cell is intentionally not executed during source preparation. Stop the cell/runtime to close the tunnel.

In [ ]:
import re, secrets, time
server_env=env.copy(); server_env['VT_API_TOKEN']=''; server=subprocess.Popen([PY,'-m','uvicorn','voice_translator.api:app','--host','0.0.0.0','--port','8000','--workers','1'],cwd=PROJECT,env=server_env)
subprocess.run(['wget','-q','-O','/content/cloudflared','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],check=True); os.chmod('/content/cloudflared',0o755)
tunnel=subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8000'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=''
for _ in range(100):
    line=tunnel.stdout.readline(); match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
    if match: url=match.group(0); break
assert url; print('Open this randomized temporary microphone frontend:',url)

## 18. Create and immediately download one evidence ZIP

In [ ]:
import zipfile
ZIP=Path('/content')/f'{EVIDENCE.name}.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as archive:
    for file in EVIDENCE.rglob('*'):
        if file.is_file(): archive.write(file,file.relative_to(EVIDENCE))
print('Downloading',ZIP); files.download(str(ZIP))